In [1]:
!pip install umap-learn bertopic gensim nltk

In [3]:
import os
import re
import gzip
import json
import subprocess

import numpy as np
import pandas as pd
import torch


from tqdm import tqdm


from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic


from sentence_transformers import SentenceTransformer


from gensim import corpora
from gensim.models import CoherenceModel

import nltk
from nltk.corpus import stopwords

import plotly.express as px
import warnings
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords", quiet=True)

RUSSIAN_STOP_WORDS = stopwords.words("russian")

In [9]:
!pip install nltk

In [4]:
DATA_PATH = "lenta-ru-news.csv"
GZ_PATH = "lenta-ru-news.csv.gz"
SAMPLE_SIZE = 20000

subprocess.run(
    ["curl", "-L", "-O", "https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz"],
            check=True
    )


with gzip.open(GZ_PATH, "rt", encoding="utf-8") as f_in:
    with open(DATA_PATH, "w", encoding="utf-8") as f_out:
        for line in f_in:
            f_out.write(line)


df = pd.read_csv(DATA_PATH, low_memory=False)


# Пустые тексты не несут информации и ломают эмбеддеры
# Слишком короткие тексты не дают достаточного контекста для кластеризации
#Для BERTopic глубокая лемматизация не обязательна на этапе эмбеддинга, так как трансформеры учитывают морфологию, но базовая очистка от шума нужна

df = df.dropna(subset=["text"])
df = df[df["text"].apply(lambda x: isinstance(x, str) and len(str(x).split()) > 15)]


df_sample = df.sample(n=SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)
texts = df_sample["text"].tolist()



#Удаляем URL: они не несят семантической нагрузки для темы
#Оставляем базовую пунктуацию, так как она может влиять на токенизацию трансформера

def clean_text(text):
    text = re.sub(r"<[^>]+>", " ", str(text))
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"[^\w\s.,!?;:\-()]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.lower()

texts_clean = [clean_text(t) for t in tqdm(texts, desc="Text cleaning")]
df_sample["text_clean"] = texts_clean

Text cleaning: 100%|██████████| 20000/20000 [00:03<00:00, 5034.08it/s]


In [5]:
# paraphrase-multilingual-MiniLM-L12-v2
# поддерживает русский язык
# MiniLM быстрее и меньше чем base-версии BERT с вроде как минимальной потерей качества
#Дообучен на задаче семантического поиска, что идеально для кластеризации документов

embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")


# Почему UMAP, а не ванильный PCA
#PCA - линейный метод, теряет нелинейные зависимости в семантическом пространстве
#UMAP же сохраняет как локальную структуру, так и глобальную, что все-таки важно для HDBSCAN

#n_neighbors=15 баланс между локальной точностью и глобальной структурой
#n_components=5
#min_dist=0.0: значение по умолчанию
#metric="cosine" - в metric learning используется кстати эта метрика, ну и она просто мастхэв


umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=SEED
)


# Почему HDBSCAN, а не K-Means - пробовали на работе и то и это и  hdbcan лучше сильно справился
# Не нужно задавать число кластеров заранее
# Умеет выделять выбросы и выделять их в отдельный кластер
# Находит кластера произвольной формы

# Параметры:
# min_cluster_size=25 меньше -получим много микро тем из 2-3 документов, что сложно интерпретировать
# - min_samples=5: на работе используем именно 5 изображений для формирования кластера (чисто эмпирически пришли к этому, хотя пробовали и другие но там по качеству было хуже). там была задача кластеризации товаров и также юзали hdbscan
# cluster_selection_method="eom" стандартный метод выбора иерархии кластеров

hdbscan_model = HDBSCAN(
    min_cluster_size=25,
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

# Векторизация для c-TF-IDF

# stop_words="russian" удаляем частые местоимения, союзы, предлоги, не несущие смысловой нагрузки
# ngram_range=(1, 3)- учитываем не только отдельные слова, но и биграммы/триграммы
#min_df=0.005 слово должно встречаться минимум в 0.5% документов кластера, отсекаем редкие опечатки - меньше шума из-за этого
#max_df=0.85 слово не должно встречаться более чем в 85% документов, отсекаем слишком общие слова - тоже упоминали в лекциях каких-то

vectorizer_model = CountVectorizer(
    stop_words=RUSSIAN_STOP_WORDS,
    ngram_range=(1, 3),
    min_df=0.005,
    max_df=0.85,
)

# nr_topics="auto": автоматическое слияние слишком похожих тем на основе косинусного сходства
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    nr_topics="auto",
    calculate_probabilities=True,
    verbose=True
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
topics, probs = topic_model.fit_transform(texts_clean)

df_sample["topic"] = topics
df_sample["prob"] = probs.max(axis=1) if probs is not None else 0.0

topic_info = topic_model.get_topic_info()
print("информация о топиках")
print(topic_info.head(10).to_markdown(index=False))

print("\n распределение топиков по количеству документов")
print(topic_info[["Topic", "Count"]].head(15).to_markdown(index=False))

2026-04-05 10:29:10,684 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/625 [00:00<?, ?it/s]

2026-04-05 10:30:30,917 - BERTopic - Embedding - Completed ✓
2026-04-05 10:30:30,920 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-05 10:31:13,468 - BERTopic - Dimensionality - Completed ✓
2026-04-05 10:31:13,471 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-05 10:31:46,480 - BERTopic - Cluster - Completed ✓
2026-04-05 10:31:46,481 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-04-05 10:33:01,009 - BERTopic - Representation - Completed ✓
2026-04-05 10:33:01,167 - BERTopic - Topic reduction - Reducing number of topics
2026-04-05 10:33:01,213 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-05 10:33:46,912 - BERTopic - Representation - Completed ✓
2026-04-05 10:33:47,086 - BERTopic - Topic reduction - Reduced number of topics from 134 to 27


информация о топиках
|   Topic |   Count | Name                                  | Representation                                                                                                             | Representative_Docs                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [7]:
# Гистограмма распределения документов по темам
fig_hist = px.histogram(
    topic_info[topic_info["Topic"] != -1],
    x="Topic", y="Count",
    title="Распределение документов по темам",
    template="plotly_white",
    labels={"Topic": "ID топика", "Count": "Число документов"}
)
fig_hist.write_html("topic_distribution.html")
fig_hist.show()

# Топ токены для каждой темы

fig_barchart = topic_model.visualize_barchart(top_n_topics=15)
fig_barchart.write_html("top_words_barchart.html")
fig_barchart.show()

# 2D проекция документов

umap_2d = UMAP(n_components=2, metric="cosine", random_state=SEED)
embeddings = embedding_model.encode(texts_clean, show_progress_bar=False)
embeddings_2d = umap_2d.fit_transform(embeddings)

fig_scatter = px.scatter(
    x=embeddings_2d[:, 0], y=embeddings_2d[:, 1],
    color=[str(t) for t in topics],
    title="2D проекция документов по темам",
    color_discrete_map={"-1": "gray"}, template="plotly_white",
    hover_data={
        "topic": topics,
        "prob": df_sample["prob"],
        "text_preview": [t[:60]+"..." for t in texts_clean]
    }
)
fig_scatter.update_traces(marker=dict(size=5, opacity=0.6))
fig_scatter.write_html("documents_2d_scatter.html")
fig_scatter.show()

# Распределение тем для выборочных текстов

sample_indices = [100, 500, 1500]
for idx in sample_indices:
    doc_probs = topic_model.transform([texts_clean[idx]])[1][0]
    top_n = 10
    topic_ids = np.argsort(doc_probs)[-top_n:][::-1]
    topic_vals = doc_probs[topic_ids]

    fig_dist = px.bar(
        x=[str(tid) for tid in topic_ids], y=topic_vals,
        labels={"x": "ID топика", "y": "Вероятность"},
        title=f"Распределение тем для документа #{idx}", template="plotly_white"
    )
    fig_dist.write_html(f"doc_{idx}_topic_dist.html")
    fig_dist.show()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-05 10:36:48,078 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-04-05 10:37:12,902 - BERTopic - Dimensionality - Completed ✓
2026-04-05 10:37:12,902 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-04-05 10:37:12,906 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-04-05 10:37:12,912 - BERTopic - Probabilities - Completed ✓
2026-04-05 10:37:12,913 - BERTopic - Cluster - Completed ✓


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-05 10:37:13,041 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-04-05 10:37:13,046 - BERTopic - Dimensionality - Completed ✓
2026-04-05 10:37:13,047 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-04-05 10:37:13,048 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-04-05 10:37:13,053 - BERTopic - Probabilities - Completed ✓
2026-04-05 10:37:13,054 - BERTopic - Cluster - Completed ✓


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-05 10:37:13,161 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-04-05 10:37:13,167 - BERTopic - Dimensionality - Completed ✓
2026-04-05 10:37:13,167 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-04-05 10:37:13,169 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-04-05 10:37:13,176 - BERTopic - Probabilities - Completed ✓
2026-04-05 10:37:13,177 - BERTopic - Cluster - Completed ✓


In [8]:
def calculate_topic_diversity(topics, model, top_n=10):
    """доля уникальных слов среди топ - N слов всех тем"""

    all_words = []
    for t in set(topics):
        if t == -1: continue
        words = [w[0] for w in model.get_topic(t)][:top_n]
        all_words.extend(words)
    return len(set(all_words)) / len(all_words) if all_words else 0.0

def calculate_umass_coherence(texts_clean, topics, model):
    """ооценивает семантическую связность слов внутри темы на основе совместной встречаемости"""

    tokenized = [t.split() for t in texts_clean]
    dictionary = corpora.Dictionary(tokenized)

    topic_words = []
    max_t = max(t for t in topics if t != -1)
    for t in range(max_t + 1):
        words = model.get_topic(t)
        if words:
            topic_words.append([w[0] for w in words])
    if not topic_words:
        return 0.0

    return CoherenceModel(
        topics=topic_words,
        texts=tokenized,
        dictionary=dictionary,
        coherence="u_mass"
    ).get_coherence()

diversity_score = calculate_topic_diversity(topics, topic_model, top_n=10)
umass_score = calculate_umass_coherence(texts_clean, topics, topic_model)

print(f"\nМетрики качества")
print(f"Topic Diversity: {diversity_score:.4f}")
print(f"UMass Coherence: {umass_score:.4f}")


Метрики качества
Topic Diversity: 0.9000
UMass Coherence: -4.1528


In [11]:
results = {
    "seed": SEED,
    "sample_size": SAMPLE_SIZE,
    "num_topics": len(set(topics) - {-1}),
    "num_outliers": sum(1 for t in topics if t == -1),
    "topic_diversity": diversity_score,
    "umass_coherence": umass_score,
    "top_topics": topic_info[topic_info["Topic"] != -1].head(5)[["Topic", "Count", "Representation"]].to_dict("records")
}

with open("results_summary.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("\nАнализ результатов")
print(f"Модель выделила {results['num_topics']} тем из {SAMPLE_SIZE} документов")
print(f"Выбросов {results['num_outliers']} ({100*results['num_outliers']/SAMPLE_SIZE:.1f}%)")


print(f"\nTopic Diversity: {diversity_score:.4f}")
print(" достаточно высокое значение 0.9 указывает на минимальное пререкрытие топ-слов между темами, + ключевые слова в разных топиках практически не потворяются, что облегчает интерпретацию")


print(f"\n UMass Coherence: {umass_score:.4f}")
print(" Значение -4.15 не очень хорошее это говорит о слабой совместной встречаемости топ-слов внутри тем в исходном корпусе возможно из-за морфологический вариативности нашего языка и из-за размытии внутренней связности внутри кластера")


print("для улучшения можно реализовать лемматиазцию, поиграться с параметрами, возможно увеличить min_clustrer_size для формирования более плотных и однородных кластеров, ну и дообучение энкодера на нашем домене новостей")


Анализ результатов
Модель выделила 26 тем из 20000 документов
Выбросов 7193 (36.0%)

Topic Diversity: 0.9000
 достаточно высокое значение 0.9 указывает на минимальное пререкрытие топ-слов между темами, + ключевые слова в разных топиках практически не потворяются, что облегчает интерпретацию

 UMass Coherence: -4.1528
 Значение -4.15 не очень хорошее это говорит о слабой совместной встречаемости топ-слов внутри тем в исходном корпусе возможно из-за морфологический вариативности нашего языка и из-за размытии внутренней связности внутри кластера
для улучшения можно реализовать лемматиазцию, поиграться с параметрами, возможно увеличить min_clustrer_size для формирования более плотных и однородных кластеров, ну и дообучение энкодера на нашем домене новостей
